In [ ]:
import sys
import os
import time
import pickle
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.utils import to_categorical
from PyQt5.QtWidgets import (QApplication, QWidget, QLabel, QComboBox, 
                             QPushButton, QTextEdit, QVBoxLayout, 
                             QHBoxLayout, QGroupBox, QFormLayout, QFileDialog)
from PyQt5.QtCore import QThread, pyqtSignal, Qt
from PyQt5.QtGui import QFont
from matplotlib.backends.backend_qt5agg import FigureCanvasQTAgg as FigureCanvas
from matplotlib.backends.backend_qt5agg import NavigationToolbar2QT as NavigationToolbar
from matplotlib.figure import Figure


# ─────────────────────────────────────────────────────────────────
# ۰. بارگذاری آفلاین دیتاست
# ─────────────────────────────────────────────────────────────────
# ─────────────────────────────────────────────────────────────────
# ۰. تابع اصلاح‌شده برای بارگذاری آفلاین یا دانلود خودکار دیتاست CIFAR-10
# ─────────────────────────────────────────────────────────────────
def load_cifar10_locally(folder_path="cifar-10-batches-py"):
    """بررسی وجود دیتاست؛ در صورت عدم وجود، آن را دانلود و استخراج می‌کند."""
    import urllib.request
    import tarfile

    # اگر پوشه دیتاست وجود نداشت، فرآیند دانلود و استخراج آغاز می‌شود
    if not os.path.exists(folder_path):
        print(f"⚠️ پوشه '{folder_path}' یافت نشد. در حال دانلود خودکار دیتاست...")
        url = "https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz"
        tar_filename = "cifar-10-python.tar.gz"
        
        # ۱. دانلود فایل فشرده
        try:
            opener = urllib.request.build_opener()
            opener.addheaders = [('User-agent', 'Mozilla/5.0')]
            urllib.request.install_opener(opener)
            
            print("📥 در حال دانلود CIFAR-10 (حدود ۱۷۰ مگابایت)... لطفاً شکیبا باشید.")
            urllib.request.urlretrieve(url, tar_filename)
            print("✅ دانلود با موفقیت انجام شد.")
        except Exception as e:
            raise RuntimeError(f"❌ خطا در دانلود دیتاست: {str(e)}\nلطفاً اتصال اینترنت خود را بررسی کنید.")
        
        # ۲. استخراج فایل فشرده
        try:
            print("📦 در حال استخراج فایل فشرده...")
            with tarfile.open(tar_filename, "r:gz") as tar:
                tar.extractall()
            print("✅ استخراج کامل شد.")
            
            # پاک کردن فایل tar.gz دانلود شده برای بهینه‌سازی فضا
            if os.path.exists(tar_filename):
                os.remove(tar_filename)
        except Exception as e:
            raise RuntimeError(f"❌ خطا در استخراج فایل دیتاست: {str(e)}")

    # بارگذاری بچ‌های آموزش (کد اصلی شما بدون تغییر)
    train_x, train_y = [], []
    for i in range(1, 6):
        p = os.path.join(folder_path, f"data_batch_{i}")
        with open(p, 'rb') as f:
            batch = pickle.load(f, encoding='bytes')
            train_x.append(batch[b'data'])
            train_y.append(batch[b'labels'])
            
    X_train = np.concatenate(train_x).reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
    y_train = np.concatenate(train_y).reshape(-1, 1)
    
    # بارگذاری داده‌های تست (کد اصلی شما بدون تغییر)
    p_test = os.path.join(folder_path, "test_batch")
    with open(p_test, 'rb') as f:
        batch = pickle.load(f, encoding='bytes')
        X_test = batch[b'data'].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
        y_test = np.array(batch[b'labels']).reshape(-1, 1)
        
    return (X_train, y_train), (X_test, y_test)

# ─────────────────────────────────────────────────────────────────
# ۱. مدل ها 
# ─────────────────────────────────────────────────────────────────
AVAILABLE_MODELS = {
    "Custom CNN (From Scratch)": {
        "build_fn": lambda: models.Sequential([
            layers.Conv2D(32, (3, 3), padding='same', activation='relu', input_shape=(32, 32, 3)),
            layers.BatchNormalization(),
            layers.MaxPooling2D((2, 2)),
            layers.Flatten(),
            layers.Dense(128, activation='relu'),
            layers.Dropout(0.3),
            layers.Dense(10, activation='softmax')
        ]),
        "input_shape": (32, 32, 3),
        "upsample": False,
        "epochs": 5
    },
    "VGG16 (Pretrained)": {
        "build_fn": lambda: tf.keras.applications.VGG16(include_top=False, weights='imagenet'),
        "input_shape": (32, 32, 3),
        "upsample": False,
        "epochs": 3
    },
    "VGG19 (Pretrained)": {
        "build_fn": lambda: tf.keras.applications.VGG19(include_top=False, weights='imagenet'),
        "input_shape": (32, 32, 3),
        "upsample": False,
        "epochs": 3
    },
    "InceptionV3 (Pretrained)": {
        "build_fn": lambda: tf.keras.applications.InceptionV3(include_top=False, weights='imagenet'),
        "input_shape": (75, 75, 3),
        "upsample": True,
        "epochs": 3
    },
    "ResNet50 (Pretrained)": {
        "build_fn": lambda: tf.keras.applications.ResNet50(include_top=False, weights='imagenet'),
        "input_shape": (32, 32, 3),
        "upsample": True, # نیاز به سایز بزرگتر دارد
        "epochs": 3
    },
    "EfficientNetB0 (Pretrained)": {
        "build_fn": lambda: tf.keras.applications.EfficientNetB0(include_top=False, weights='imagenet'),
        "input_shape": (32, 32, 3),
        "upsample": False,
        "epochs": 3
    },
    "MobileNetV2 (Pretrained)": {
        "build_fn": lambda: tf.keras.applications.MobileNetV2(include_top=False, weights='imagenet'),
        "input_shape": (32, 32, 3),
        "upsample": False,
        "epochs": 3
    },
    "Hybrid Ensemble (VGG16 + InceptionV3) - Online": {
        "is_ensemble": True,
        "epochs": 3
    }
}

WEIGHT_EXTENSIONS = (".weights.h5",)

def get_weight_files():
    base_dir = os.path.dirname(os.path.abspath(sys.argv[0]))
    weights = {}
    for file in os.listdir(base_dir):
        if file.endswith(WEIGHT_EXTENSIONS):
            weights[file] = os.path.join(base_dir, file)
    return weights

# ─────────────────────────────────────────────────────────────────
# ۲. کلاس ترد پس‌زمینه برای محاسبات سنگین و استخراج معیارها
# ─────────────────────────────────────────────────────────────────
class TrainingWorker(QThread):
    log_signal = pyqtSignal(str)       
    finished_signal = pyqtSignal(str, dict)  # ارسال خلاصه متنی و داده‌های ارزیابی

    def __init__(self, model_name, weight_file=None, custom_epochs=None):
        super().__init__()
        self.model_name = model_name
        self.weight_file = weight_file
        self.custom_epochs = custom_epochs

    def run(self):
        try:
            self.log_signal.emit("⏳ بارگذاری و نرمال‌سازی داده‌های CIFAR-10...")
            (X_train, y_train), (X_test, y_test) = load_cifar10_locally("cifar-10-batches-py")
            
            X_train = X_train.astype('float32') / 255.0
            X_test = X_test.astype('float32') / 255.0
            y_train_cat = to_categorical(y_train, 10)
            y_test_cat = to_categorical(y_test, 10)

            X_train, y_train_cat = X_train[:15000], y_train_cat[:15000]
            X_test_sub, y_test_cat_sub = X_test[:3000], y_test_cat[:3000]
            y_test_sub = y_test[:3000]

            class GuiLogger(keras.callbacks.Callback):
                def __init__(self, emitter): self.emitter = emitter
                def on_epoch_end(self, epoch, logs=None):
                    self.emitter.emit(f"Epoch {epoch+1}: Loss={logs['loss']:.4f}, Acc={logs['accuracy']:.4f} | Val_Loss={logs['val_loss']:.4f}, Val_Acc={logs['val_accuracy']:.4f}")

            model_cfg = AVAILABLE_MODELS[self.model_name]
            if self.custom_epochs is not None:
                epochs = self.custom_epochs
            else:
                epochs = model_cfg["epochs"]
            
            self.log_signal.emit(f"🏗️ در حال پیکربندی مدل انتخابی: {self.model_name}...")
            start_time = time.time()

            if model_cfg.get("is_ensemble"):
                input_layer = layers.Input(shape=(32, 32, 3))
                resized = layers.UpSampling2D(size=(3, 3))(input_layer)
                
                base1 = tf.keras.applications.VGG16(input_shape=(32, 32, 3), include_top=False, weights='imagenet')
                base2 = tf.keras.applications.InceptionV3(input_shape=(96, 96, 3), include_top=False, weights='imagenet')
                base1.trainable, base2.trainable = False, False
                
                f1 = layers.GlobalAveragePooling2D()(base1(input_layer))
                f2 = layers.GlobalAveragePooling2D()(base2(resized))
                merged = layers.concatenate([f1, f2])
                x = layers.Dense(256, activation='relu')(merged)
                output_layer = layers.Dense(10, activation='softmax')(x)
                model = models.Model(inputs=input_layer, outputs=output_layer)
            else:
                inputs = layers.Input(shape=(32, 32, 3))
                x = inputs
                if model_cfg["upsample"]:
                    size_multiplier = int(np.ceil(model_cfg["input_shape"][0] / 32))
                    x = layers.UpSampling2D(size=(size_multiplier, size_multiplier))(x)
                
                base = model_cfg["build_fn"]()
                if isinstance(base, models.Sequential):
                    model = base
                else:
                    base.trainable = False
                    x = base(x)
                    x = layers.GlobalAveragePooling2D()(x)
                    x = layers.Dense(256, activation='relu')(x)
                    outputs = layers.Dense(10, activation='softmax')(x)
                    model = models.Model(inputs=inputs, outputs=outputs)

            model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
            if self.weight_file:
                filename = os.path.basename(self.weight_file).lower()
                model_token = (self.model_name.split("(")[0].strip().lower())
                if model_token not in filename:
                    self.finished_signal.emit(f"❌ فایل وزن انتخاب شده برای مدل [{self.model_name}] نیست.", {})
                    return
                try:
                    self.log_signal.emit(f"📥 در حال بارگذاری وزن‌های سفارشی محلی:\n{self.weight_file}")
                    
                    # اضافه کردن کلماتی کلیدی skip_mismatch و by_name برای جلوگیری از کرش
                    model.load_weights(self.weight_file, skip_mismatch=True, by_name=True)
                    
                    self.log_signal.emit("✅ وزن‌های محلی (لایه‌های سازگار) با موفقیت بارگذاری شدند.")
                except Exception as e:
                    self.finished_signal.emit(f"❌ وزن انتخاب شده با معماری مدل سازگار نیست.\n\n{str(e)}", {})
                    return            
            else:
                # حالتی که کاربر هیچ وزنی انتخاب نکرده و می‌خواهد از وزن‌های پیش‌فرض آنلاین استفاده کند
                self.log_signal.emit("🌐 فایل وزنی انتخاب نشد. سیستم از وزن‌های آنلاین Imagenet (در صورت وجود در پوشه .keras یا دانلود خودکار) استفاده می‌کند...")           

            total_params = model.count_params()
            trainable_params = np.sum([keras.backend.count_params(w) for w in model.trainable_weights])
            
            self.log_signal.emit(f"📊 مشخصات شبکه:\n   کل پارامترها: {total_params:,}\n   پارامترهای قابل آموزش: {trainable_params:,}")
            self.log_signal.emit(f"🚀 شروع فرآیند آموزش به مدت {epochs} اپوک...")
            
            history = model.fit(X_train, y_train_cat, epochs=epochs, batch_size=128, 
                                validation_data=(X_test_sub, y_test_cat_sub), 
                                callbacks=[GuiLogger(self.log_signal)], verbose=0)

            self.log_signal.emit("💾 در حال ذخیره و پشتیبان‌گیری از وزن‌های آموزش دیده...")
            clean_name = self.model_name.split("(")[0].strip().replace(" ", "_").lower()
            model.save_weights(f"{clean_name}_trained.weights.h5")
            self.log_signal.emit("✅ فایل وزن با موفقیت در کنار اسکریپت ذخیره شد.")
            
            training_duration = time.time() - start_time
            self.log_signal.emit("📊 در حال اجرای محاسبات ارزیابی نهایی ...")
            
            loss, accuracy = model.evaluate(X_test_sub, y_test_cat_sub, verbose=0)
            
            y_pred = model.predict(X_test_sub, verbose=0)
            y_pred_classes = np.argmax(y_pred, axis=1)
            
            cm = confusion_matrix(y_test_sub, y_pred_classes)
            report = classification_report(y_test_sub, y_pred_classes, target_names=[
                'Airplane', 'Automobile', 'Bird', 'Cat', 'Deer', 'Dog', 'Frog', 'Horse', 'Ship', 'Truck'
            ])

            eval_data = {
                "history": history.history,
                "cm": cm,
                "report": report,
                "duration": training_duration,
                "total_params": total_params,
                "X_test_images": X_test_sub,       # ارسال آرایه تصاویر تست
                "y_true": y_test_sub.flatten(),    # برچسب‌های واقعی
                "y_pred": y_pred_classes           # برچسب‌های پیش‌بینی شده
            }
            
            summary = f"✅ ارزیابی پایان یافت.\n🎯 Accuracy نهایی: {accuracy*100:.2f}%\n📉 Loss نهایی: {loss:.4f}"
            self.finished_signal.emit(summary, eval_data)
            
        except Exception as e:
            self.finished_signal.emit(f"❌ خطا در اجرای فرآیند: {str(e)}", {})

# ─────────────────────────────────────────────────────────────────
# ۳. رابط کاربری
# ─────────────────────────────────────────────────────────────────
class DeepLearningGui(QWidget):
    def __init__(self):
        super().__init__()
        self.weight_files = {}
        self.init_ui()

    def init_ui(self):
        self.setWindowTitle("سیستم مدیریت پلاگین و ارزیابی هوش مصنوعی")

        self.resize(1440, 900)
        
        self.setStyleSheet("""
            QWidget {
                background-color: #f8f9fa;
                color: #212529;
                font-family: 'Segoe UI', Tahoma, Arial;
            }
            QGroupBox {
                border: 1px solid #dee2e6;
                border-radius: 6px;
                margin-top: 15px;
                font-weight: bold;
                font-size: 13px;
                color: #495057;
                background-color: #ffffff;
            }
            QGroupBox::title {
                subcontrol-origin: margin;
                left: 10px;
                padding: 0 5px;
            }
            QComboBox {
                background-color: #ffffff;
                border: 1px solid #ced4da;
                border-radius: 4px;
                padding: 6px;
                min-width: 250px;
                color: #212529;
            }
            QComboBox:hover {
                border-color: #0d6efd;
            }
            QPushButton {
                background-color: #0d6efd;
                color: white;
                border: none;
                border-radius: 4px;
                padding: 10px 20px;
                font-weight: bold;
                font-size: 13px;
            }
            QPushButton:hover {
                background-color: #0b5ed7;
            }
            QPushButton:disabled {
                background-color: #e9ecef;
                color: #6c757d;
            }
            QTextEdit {
                background-color: #f1f3f5;
                border: 1px solid #ced4da;
                border-radius: 4px;
                font-family: 'Consolas', monospace;
                font-size: 12px;
                color: #212529;
            }
            /* ─── استایل اختصاصی و فوق‌العاده مدرن برای دکمه کامپایل ─── */
            QPushButton#btn_compile_hero {
                background: qlineargradient(x1:0, y1:0, x2:1, y2:1, stop:0 #0d6efd, stop:1 #0a58ca); /* گرادیان آبی جذاب */
                color: white;
                border: 1px solid #0a58ca;
                border-radius: 6px;
                padding: 12px 24px;
                font-size: 14px;
                font-weight: bold;
                letter-spacing: 0.5px;
            }
            
            /* حالت شناور شدن موس روی دکمه (Hover) */
            QPushButton#btn_compile_hero:hover {
                background: qlineargradient(x1:0, y1:0, x2:1, y2:1, stop:0 #0b5ed7, stop:1 #04419c);
                border-color: #04419c;
                font-size: 14.5px; /* یک مقدار بسیار کم بزرگتر می‌شود تا حالت انیمیشنی به خود بگیرد */
            }

            /* حالت کلیک شدن روی دکمه (Pressed) */
            QPushButton#btn_compile_hero:pressed {
                background-color: #0a58ca;
                padding-top: 13px; /* افکت فشرده شدن ناخودآگاه کلید به سمت پایین */
                padding-bottom: 11px;
            }

            /* حالت غیرفعال شدن دکمه در زمان اجرای آموزش (Disabled) */
            QPushButton#btn_compile_hero:disabled {
                background: #e9ecef;
                color: #adb5bd;
                border: 1px solid #dee2e6;
            }
        """)

        window_layout = QHBoxLayout()
        
        left_panel = QVBoxLayout()

        header_label = QLabel("میز کار ارزیابی و مانیتورینگ عملکرد مدل های CNN")
        header_label.setAlignment(Qt.AlignCenter)
        header_label.setStyleSheet("font-size: 18px; font-weight: bold; color: #1a1a1a; padding: 5px;")
        left_panel.addWidget(header_label)

        # پنل بالا: مدیریت معماری‌ها با لایوت کاملاً افقی
        config_box = QGroupBox("مدیریت معماری‌ها")
        config_layout = QHBoxLayout() 

        config_box.setLayoutDirection(Qt.RightToLeft)
        
        self.model_selector = QComboBox()
        self.model_selector.addItems(list(AVAILABLE_MODELS.keys()))

        self.weight_selector = QComboBox()

        self.epoch_selector = QComboBox()
        self.epoch_selector.addItems(["1", "3", "5", "10", "20", "50"])
        self.epoch_selector.setCurrentText("3")

        self.btn_refresh_weights = QPushButton("🔄 بروزرسانی وزن‌ها")
        self.btn_refresh_weights.clicked.connect(self.load_weight_files)
        self.btn_refresh_weights.setStyleSheet("padding: 6px 12px; font-size: 11px;")
        #self.btn_refresh_weights.hide() # مخفی بودن در شروع برنامه

        config_layout.addWidget(QLabel("مدل:"))
        config_layout.addWidget(self.model_selector)
        
        config_layout.addWidget(QLabel("وزن:"))
        config_layout.addWidget(self.weight_selector)
        
        config_layout.addWidget(QLabel("اپوک:"))
        config_layout.addWidget(self.epoch_selector)
        
        config_layout.addWidget(self.btn_refresh_weights) # سهم دکمه (۲ واحد)
        
        # تنظیم فواصل داخلی برای تمیز بودن ظاهر
        config_layout.setContentsMargins(15, 15, 15, 15)
        config_layout.setSpacing(15)
        
        config_box.setLayout(config_layout)
        
        left_panel.addWidget(config_box)

        self.btn_start = QPushButton("🚀 کامپایل، آموزش و ارزیابی تخصصی")
        self.btn_start.setObjectName("btn_compile_hero")
        self.btn_start.clicked.connect(self.start_training)
        left_panel.addWidget(self.btn_start)

# ─── شروع بخش اصلاح‌شده و یکپارچه خروجی‌ها ───
        bottom_layout = QHBoxLayout()
        
        # ۱. باکس ستون اول: کنسول و مانیتورینگ سیستم (سمت چپ)
        console_box = QGroupBox("کنسول و مانیتورینگ سیستم")
        console_inner = QVBoxLayout()
        self.log_viewer = QTextEdit()
        self.log_viewer.setReadOnly(True)
        console_inner.addWidget(self.log_viewer)
        console_box.setLayout(console_inner)
        
        # ۲. باکس ستون دوم: گزارش ارزیابی نهایی (وسط)
        report_box = QGroupBox("گزارش ارزیابی نهایی (Metrics Report)")
        report_inner = QVBoxLayout()
        self.report_viewer = QTextEdit()
        self.report_viewer.setReadOnly(True)
        self.report_viewer.setStyleSheet("background-color: #ffffff; color: #111111;")
        report_inner.addWidget(self.report_viewer)
        report_box.setLayout(report_inner)

        # ۳. باکس ستون سوم: نمودارهای بصری‌سازی کارایی مدل (سمت راست)
        plot_box = QGroupBox("نمودارهای بصری‌سازی کارایی مدل")
        plot_inner = QVBoxLayout()
        
        # ایجاد آبجکت Figure و اختصاص Canvas به آن (کدهای قدیمی از انتهای متد به اینجا منتقل شدند)
        self.fig = Figure(figsize=(6, 8), dpi=100)
        self.canvas = FigureCanvas(self.fig)
        #self.toolbar = NavigationToolbar(self.canvas, self)
        
        # اضافه کردن تولبار و کانواس مستقیماً به داخل باکس ستون سوم
        #plot_inner.addWidget(self.toolbar)
        plot_inner.addWidget(self.canvas)
        plot_box.setLayout(plot_inner)

        # اضافه کردن هر ۳ ستون به لایوت پایینی با نسبت‌های طلایی (۳۰٪، ۳۰٪، ۴۰٪)
         
        bottom_layout.addWidget(report_box, 40)   
        bottom_layout.addWidget(plot_box, 40)     
        bottom_layout.addWidget(console_box, 20) 
        
        # اضافه کردن لایوت ۳ ستونه به پنل اصلی سمت چپ
        left_panel.addLayout(bottom_layout)
        
        # تنظیم لایوت اصلی پنجره (حالا کل لایوت پنجره فقط شامل left_panel به صورت تمام صفحه است)
        main_window_layout = QVBoxLayout()
        main_window_layout.addLayout(left_panel)
        self.setLayout(main_window_layout)
        
        self.load_weight_files()
        self.log_viewer.append("✦ سیستم آماده است. مدل و وزن مرتبط مورد نظر را انتخاب و دکمه شروع را کلیک کنید...")
        # ─── پایان متد init_ui ───

    def load_weight_files(self):
        base_dir = os.getcwd()
        self.weight_files = {}
        self.weight_selector.clear()
        self.weight_selector.addItem("None")
        for file in os.listdir(base_dir):
            if file.lower().endswith((".weights.h5", ".h5")):
                self.weight_files[file] = os.path.join(base_dir, file)
                self.weight_selector.addItem(file)
        self.log_viewer.append(f"FOUND {len(self.weight_files)} WEIGHT FILES")
        if self.weight_files:
            self.log_viewer.append("📦 Weight Files:")
            for name in self.weight_files.keys():
                self.log_viewer.append(f"   • {name}")
        else:
            self.log_viewer.append("⚠️ هیچ فایل وزنی پیدا نشد.")
        
    def start_training(self):
        selected_model = self.model_selector.currentText()
        selected_weight = self.weight_selector.currentText()
        selected_epochs = int(self.epoch_selector.currentText())
        
        weight_path = None
        
        if selected_weight != "None":
            weight_path = self.weight_files[selected_weight]
        
        self.btn_start.setEnabled(False)
        self.model_selector.setEnabled(False)
        self.btn_refresh_weights.setEnabled(False)
        self.weight_selector.setEnabled(False)
        self.epoch_selector.setEnabled(False)
        
        
        self.log_viewer.clear()
        self.report_viewer.clear()
        self.log_viewer.append(f"❖ ترد پس‌زمینه برای مدل [{selected_model}] فعال شد.")

        self.worker = TrainingWorker(selected_model, weight_path, selected_epochs)
        self.worker.log_signal.connect(self.update_logs)
        self.worker.finished_signal.connect(self.training_finished)
        self.worker.start()

    def update_logs(self, text):
        self.log_viewer.append(text)
        self.log_viewer.ensureCursorVisible()

    def training_finished(self, final_result, eval_data):
        self.log_viewer.append("\n" + "="*40 + "\n" + final_result)
        self.log_viewer.ensureCursorVisible()
        
        if eval_data:
            report_text = f"⏱️ زمان کل فرآیند: {eval_data['duration']:.2f} ثانیه\n"
            report_text += f"🎛️ تعداد کل پارامترها: {eval_data['total_params']:,}\n"
            report_text += f"\n📊 Classification Report:\n\n{eval_data['report']}"
            self.report_viewer.setText(report_text)
            
            # فراخوانی متد جدید برای پلات زدن در بوم داخلی
            self.plot_visualizations(eval_data)

        self.btn_start.setEnabled(True)
        self.model_selector.setEnabled(True)
        self.epoch_selector.setEnabled(True)
        self.weight_selector.setEnabled(True)
        self.btn_refresh_weights.setEnabled(True)
        
    def plot_visualizations(self, eval_data):
        """ تولید پلات‌های تعبیه شده داخل خود بوم GUI شامل تصاویر پیش‌بینی شده """
        history = eval_data["history"]
        cm = eval_data["cm"]
        
        # ۱. ابتدا پلات‌های قبلی را پاک می‌کنیم
        self.fig.clear()
        self.canvas.draw()
        
        # ۲. ساختار جدید گرید (به جای ساب‌پلات ساده، چیدمان را پیشرفته‌تر می‌کنیم)
        # یک شبکه با ۳ سطر ایجاد می‌کنیم: سطر اول چارت، سطر دوم ماتریس، سطر سوم تصاویر
        ax1 = self.fig.add_subplot(3, 1, 1)
        ax2 = self.fig.add_subplot(3, 1, 2)
        
        # ─── پلات اول: نمودار Accuracy ───
        ax1.plot(history['accuracy'], label='Train Acc', color='#0d6efd', linewidth=2)
        ax1.plot(history['val_accuracy'], label='Val Acc', color='#ffc107', linewidth=2)
        ax1.set_title('Accuracy Curve', fontsize=9, fontweight='bold')
        ax1.set_xlabel('Epochs', fontsize=8)
        ax1.set_ylabel('Accuracy', fontsize=8)
        ax1.legend(fontsize=8)
        ax1.grid(True, linestyle='--', alpha=0.5)
        
        # ─── پلات دوم: ماتریس آشفتگی ───
        class_names = ['Plane', 'Car', 'Bird', 'Cat', 'Deer', 'Dog', 'Frog', 'Horse', 'Ship', 'Truck']
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names, cbar=False, ax=ax2, annot_kws={"size": 8})
        ax2.set_title('Confusion Matrix', fontsize=9, fontweight='bold')
        ax2.tick_params(labelsize=8)
        
        # ─── بخش جدید: پلات سوم برای نمایش تصاویر نمونه ───
        # انتخاب چند تصویر اول از داده‌های تست برای نمایش (مثلاً ۵ تصویر)
        num_images_to_show = 5
        X_imgs = eval_data["X_test_images"][:num_images_to_show]
        y_true = eval_data["y_true"][:num_images_to_show]
        y_pred = eval_data["y_pred"][:num_images_to_show]
        
        # برای اینکه تصاویر در یک ردیف در سطر سوم جا شوند، یک دایره گرید فرعی می‌سازیم
        for i in range(num_images_to_show):
            # ایجاد ۵ ساب‌پلات کوچک در کنار هم در قالب سطر سوم (Grid دست‌ساز)
            ax_img = self.fig.add_subplot(3, num_images_to_show, (2 * num_images_to_show) + i + 1)
            
            # نمایش تصویر (دیتاست CIFAR-10 ابعاد 32x32x3 دارد)
            ax_img.imshow(X_imgs[i])
            ax_img.axis('off') # حذف محورهای مختصات تصویر
            
            # تشخیص درست یا غلط بودن پیش‌بینی برای تعیین رنگ متن
            true_name = class_names[y_true[i]]
            pred_name = class_names[y_pred[i]]
            color = 'green' if y_true[i] == y_pred[i] else 'red'
            
            # نوشتن متن برچسب بالای هر تصویر
            ax_img.set_title(f"T: {true_name}\nP: {pred_name}", color=color, fontsize=8, fontweight='bold')

        # تنظیم فاصله و بروزرسانی تغییرات روی بوم GUI
        self.fig.tight_layout()
        self.canvas.draw()

# اجرای ساختار نهایی
if __name__ == "__main__":
    app = QApplication(sys.argv)
    gui = DeepLearningGui()
    gui.show()
    sys.exit(app.exec_() if hasattr(app, 'exec') else app.exec_())

⚠️ پوشه 'cifar-10-batches-py' یافت نشد. در حال دانلود خودکار دیتاست...
📥 در حال دانلود CIFAR-10 (حدود ۱۷۰ مگابایت)... لطفاً شکیبا باشید.
✅ دانلود با موفقیت انجام شد.
📦 در حال استخراج فایل فشرده...


C:\Users\Hossein\AppData\Local\Temp\ipykernel_15436\383231598.py:56: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall()


✅ استخراج کامل شد.


C:\Users\Hossein\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
